# Cross-Validation Strategies — when shuffling is the bug

## The assumption underneath everything so far

Every cross-validation you have run has quietly assumed your rows are **independent** — that
knowing one row tells you nothing about another, so it does not matter which fold each lands
in.

That assumption is often false, and when it fails, cross-validation stops measuring
generalisation and starts measuring memorisation. The score goes *up*, which is why nobody
notices.

Two common ways it fails:

- **Grouped rows** — several rows come from the same source: the same patient, the same
  customer, the same family.
- **Ordered rows** — the data has a time order, and predicting the past from the future is
  not a thing you will ever get to do in production.

This notebook covers the splitter for each.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (
    GroupKFold,
    KFold,
    LeaveOneGroupOut,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RSEED = 42

## Part 1 — grouped data

The Titanic data has a grouping structure hiding in plain sight: **the ticket number**.
Families and travelling parties bought one ticket between them.

In [ ]:
titanic = pd.read_csv("data/titanic.csv")

groups = titanic["Ticket"]
sizes = groups.value_counts()

print(f"{len(titanic)} passengers on {groups.nunique()} tickets")
print(f"{(sizes > 1).sum()} tickets cover more than one passenger; largest party is {sizes.max()}")

Why that matters: people travelling together largely **survived or died together**. They
shared a cabin location, they were escorted to the same lifeboat, they waited for each other.

So if a mother is in the training set and her daughter is in the test set, the model can
score well on the daughter by having memorised the mother. That is not a skill it will have
on a genuinely new passenger.

In [ ]:
# how strongly does one party member's outcome predict another's?
party_survival = titanic.groupby("Ticket")["Survived"].agg(["mean", "count"])
parties = party_survival[party_survival["count"] > 1]

unanimous = ((parties["mean"] == 0) | (parties["mean"] == 1)).mean()
print(f"of the {len(parties)} multi-passenger parties, {unanimous:.1%} shared the same fate entirely")

In [ ]:
features = titanic[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]
y = titanic["Survived"]

preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
     make_column_selector(dtype_include="number")),
    (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")),
     make_column_selector(dtype_exclude="number")),
)
model = make_pipeline(preprocessor, RandomForestClassifier(n_estimators=200, random_state=RSEED, n_jobs=-1))

naive = cross_val_score(model, features, y, cv=StratifiedKFold(5, shuffle=True, random_state=RSEED))
grouped = cross_val_score(model, features, y, cv=GroupKFold(n_splits=5), groups=groups)

print(f"StratifiedKFold (parties split across folds) : {naive.mean():.4f} ± {naive.std():.4f}")
print(f"GroupKFold      (parties kept together)      : {grouped.mean():.4f} ± {grouped.std():.4f}")
print(f"\ninflation from leaking parties across folds  : {naive.mean() - grouped.mean():+.4f}")

The naive estimate is the optimistic one, and the gap is the size of the lie you would have
told yourself.

`GroupKFold` guarantees that **every row from a group lands in the same fold**. You pass the
group labels alongside the data:

```python
cross_val_score(model, X, y, cv=GroupKFold(n_splits=5), groups=groups)
```

The `groups` argument is not optional and is not inferred — you have to know your data well
enough to name the grouping column.

### `LeaveOneGroupOut`

When you have a small number of large, meaningful groups — five hospitals, three factories,
four sensors — you often want one fold per group. That answers a sharper question: *does the
model transfer to a site it has never seen?*

In [ ]:
# group Titanic passengers by where they boarded — three ports, one fold each
ports = titanic["Embarked"].fillna("Unknown")
logo = LeaveOneGroupOut()

print(f"{logo.get_n_splits(groups=ports)} folds, one per port\n")

for train_idx, test_idx in logo.split(features, y, groups=ports):
    held_out = ports.iloc[test_idx].unique()[0]
    model.fit(features.iloc[train_idx], y.iloc[train_idx])
    score = model.score(features.iloc[test_idx], y.iloc[test_idx])
    print(f"trained without {held_out:<8} → scores {score:.4f} on it ({len(test_idx):>3} passengers)")

Per-group scores are worth reading individually rather than averaged. A group the model does
badly on is telling you something specific — that site is different, and it is worth finding
out how. Here it scores 0.65 on Queenstown against 0.79 on Southampton.

**And note the last row.** The "Unknown" group has two passengers and scores a perfect
1.0000, which means nothing at all. `LeaveOneGroupOut` will happily hand you a fold of size
two; always print the group sizes next to the scores, and ignore the tiny ones.

## Part 2 — ordered data

> **A note on the data.** No dataset in this course has a genuine time index, so this section
> uses a small synthetic series. The mechanism is what matters, and the failure it prevents
> is real and expensive.

Consider a series with a trend and a weekly cycle — daily sales, server load, energy demand.

In [ ]:
n_days = 365
day = np.arange(n_days)
rng = np.random.default_rng(RSEED)

trend = 0.05 * day
weekly = 2.5 * np.sin(2 * np.pi * day / 7)
noise = rng.normal(0, 1.0, n_days)
demand = 20 + trend + weekly + noise

series = pd.DataFrame({
    "day_of_week": day % 7,
    "day_index": day,
    "lag_1": np.r_[np.nan, demand[:-1]],
    "lag_7": np.r_[[np.nan] * 7, demand[:-7]],
    "demand": demand,
}).dropna()

X_ts = series.drop(columns="demand")
y_ts = series["demand"]

plt.figure(figsize=(11, 3.2))
plt.plot(series["day_index"], y_ts, lw=1, color="#F5821F")
plt.xlabel("day")
plt.ylabel("demand")
plt.title("A year of daily demand: rising trend, weekly cycle");

### What shuffling does here

`KFold(shuffle=True)` puts day 200 in the training set and day 199 in the test set. The model
gets to see tomorrow while predicting today — and `lag_1` makes that trivially easy.

In production you will never have tomorrow. The score is fiction.

In [ ]:
ts_model = make_pipeline(StandardScaler(), LinearRegression())

shuffled = cross_val_score(ts_model, X_ts, y_ts, cv=KFold(5, shuffle=True, random_state=RSEED), scoring="r2")
ordered = cross_val_score(ts_model, X_ts, y_ts, cv=TimeSeriesSplit(n_splits=5), scoring="r2")

print(f"KFold(shuffle=True)  : {shuffled.mean():.4f} ± {shuffled.std():.4f}   ← sees the future")
print(f"TimeSeriesSplit      : {ordered.mean():.4f} ± {ordered.std():.4f}   ← honest")

### How `TimeSeriesSplit` splits

It always trains on the **past** and tests on the **future**. The training window grows with
each fold; the test window is always the block immediately after it.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

fig, ax = plt.subplots(figsize=(11, 3))
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_ts)):
    ax.scatter(train_idx, [fold] * len(train_idx), c="#2d6a9f", s=6, marker="_", linewidths=6)
    ax.scatter(test_idx, [fold] * len(test_idx), c="#F5821F", s=6, marker="_", linewidths=6)
    print(f"fold {fold + 1}: train on days 0–{train_idx.max():<4} test on days "
          f"{test_idx.min()}–{test_idx.max()}")

ax.set_yticks(range(5))
ax.set_yticklabels([f"fold {i + 1}" for i in range(5)])
ax.set_xlabel("day index")
ax.set_title("TimeSeriesSplit — blue = train, orange = test")
plt.tight_layout()
plt.show()

Two consequences of that shape worth knowing:

- **The folds are not equally informative.** Fold 1 trains on 60 days, fold 5 on 300. Early
  folds will score worse simply because they had less history, so the mean across folds is
  pessimistic relative to what a model trained on everything would do.
- **No shuffling, ever.** There is no `shuffle` parameter, because shuffling would defeat the
  entire purpose.

## Choosing a splitter

| Your data | Splitter | The failure it prevents |
|---|---|---|
| independent rows, balanced classes | `KFold(shuffle=True)` | — |
| independent rows, **imbalanced** classes | `StratifiedKFold` | a fold with too few positives |
| rows **grouped** by patient / customer / family | `GroupKFold` | memorising a group member |
| a few large, meaningful groups | `LeaveOneGroupOut` | not knowing whether it transfers to a new site |
| rows **ordered in time** | `TimeSeriesSplit` | predicting the past from the future |

**The question to ask before every split: what would make two rows in this dataset non-independent?**
If you can answer it, you know which splitter you need.

> **Where this comes from.** The Bias-Variance lesson introduced K-fold and noted that
> shuffling mattered on the wine data. This notebook is the other half of that: the cases
> where shuffling is not just unhelpful but actively wrong.